# Unsupervised Anomaly Detection for Concrete Surface Crack Inspection (Direction A.1)

**Author:** Lam Huynh Khang — SE200666 — FPT University HCMC  
**Supervisor:** Nguyen Xuan Huy, Lecturer, FPT University  

This notebook implements the complete experimental pipeline for **Direction A.1**:
Evaluating and comparing **three unsupervised visual anomaly detection methods** across **three concrete and surface crack datasets**.

### Unsupervised Methods compared:
1. **SPADE (Unsupervised Baseline):** K-NN search over a full memory bank of neighborhood-aware patch features.
2. **PaDiM (Unsupervised Baseline):** Multivariate Gaussian distribution estimation per patch position.
3. **PatchCore (Proposed Method):** Optimized patch memory bank using coreset subsampling (evaluated at 10% coreset ratio with ResNet-18 and ResNet-50 backbones).

### Datasets Evaluated:
1. **Mendeley Surface Crack Detection:** 40,000 images, widely used for concrete crack classification.
2. **Structural Defects Network (SDNET) 2018:** 56,000 images of concrete walls, bridge decks, and pavements.
3. **DeepCrack:** A pixel-level annotated crack dataset (used for cross-dataset generalization testing).

## 1. Setup & Dependencies
Clone the repository, install required libraries, and install the `patchcore` package in editable mode.

In [ ]:
# Remove old clone if any, clone latest version from repo
!rm -rf patchcore-concrete-crack-inspection
!git clone https://github.com/Welkie/patchcore-concrete-crack-inspection.git
%cd patchcore-concrete-crack-inspection

# Install required libraries and patchcore package
!pip install timm faiss-gpu scikit-image scikit-learn matplotlib tqdm pillow click pandas seaborn opencv-python-headless
!pip install -e .

## 2. Verify Environment & Imports
Verify that custom dataset classes and the unsupervised methods (PatchCore, PaDiM) can be imported successfully.

In [ ]:
import os
import sys
import torch

sys.path.append(os.path.abspath('src'))
sys.path.append(os.path.abspath('bin'))

print("PyTorch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))

# Verify imports
from patchcore.datasets.concrete import ConcreteDataset
from patchcore.datasets.sdnet import SDNETDataset
from patchcore.datasets.deepcrack import DeepCrackDataset
from patchcore.padim import PaDiM
from patchcore.patchcore import PatchCore

print("\nAll modules successfully imported!")

## 3. Locate Kaggle Datasets
Locate and verify all three required datasets on Kaggle.

In [ ]:
import os

def find_dataset(folder_keywords):
    possible_roots = ["/kaggle/input", "/kaggle/input/datasets"]
    for root in possible_roots:
        if os.path.exists(root):
            # Search subdirectories recursively up to level 2
            for level1 in os.listdir(root):
                p1 = os.path.join(root, level1)
                if os.path.isdir(p1):
                    if any(kw.lower() in level1.lower() for kw in folder_keywords):
                        return p1
                    for level2 in os.listdir(p1):
                        p2 = os.path.join(p1, level2)
                        if os.path.isdir(p2):
                            if any(kw.lower() in level2.lower() for kw in folder_keywords):
                                return p2
    return None

# 1. Surface Crack Detection
sc_path = find_dataset(["surface-crack-detection", "surface_crack_detection", "surface-crack"])
# 2. SDNET2018
sd_path = find_dataset(["structural-defects-network", "concrete-crack-images", "sdnet2018", "sdnet"])
# 3. DeepCrack
dc_path = find_dataset(["deepcrack-dataset", "deepcrack"])

print("Surface Crack Path:", sc_path)
print("SDNET2018 Path:", sd_path)
print("DeepCrack Path:", dc_path)

if not (sc_path and sd_path and dc_path):
    print("\n[WARNING] Some datasets could not be located. Please make sure all 3 datasets are added to your Kaggle Notebook input!")
else:
    print("\nAll datasets verified and ready!")

## 4. Run Direction A.1 Experiments
Run `bin/run_experiments.py` to evaluate SPADE, PaDiM, and PatchCore on the three datasets.

In [ ]:
# Execute the experiment script passing paths for all three datasets
!python bin/run_experiments.py \
    --surface_crack_path "{sc_path}" \
    --sdnet_path "{sd_path}" \
    --deepcrack_path "{dc_path}" \
    --results_path results \
    --gpu 0

## 5. View Quantitative Results
Load the compiled results CSV containing details for all datasets and unsupervised models.

In [ ]:
import pandas as pd
import os
from IPython.display import display

csv_path = "results/unsupervised_comparison_results.csv"
if os.path.exists(csv_path):
    df = pd.read_csv(csv_path)
    print("=" * 80)
    print("UNSUPERVISED COMPARISON RESULTS ACROSS DATASETS (Direction A.1)")
    print("=" * 80)
    display(df)
else:
    print("Results CSV not found. Please run the experiments first.")

## 6. Performance Evaluation Plots
Display comparative charts for detection accuracy (AUROC), localization accuracy (Pixel AUROC), and speed/latency.

In [ ]:
from PIL import Image
import os
import matplotlib.pyplot as plt

plots = [
    ("results/plot_unsupervised_auroc_comparison.png", "Image-level Detection Performance (AUROC) Comparison"),
    ("results/plot_unsupervised_localization_comparison.png", "Pixel-level Anomaly Localization (Pixel AUROC) Comparison"),
    ("results/plot_unsupervised_latency_comparison.png", "Average Inference Latency (ms/image) Comparison"),
]

for path, title in plots:
    if os.path.exists(path):
        print(f"\n{'='*60}\n{title}\n{'='*60}")
        img = Image.open(path)
        plt.figure(figsize=(10, 6))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Plot skipped (not found): {path}")

## 7. Anomaly Heatmap Visualizations
Display the qualitative anomaly localization heatmaps for all three datasets.

In [ ]:
visuals = [
    ("results/visual_localization_surface_crack.png", "Mendeley Surface Crack Localization Heatmaps"),
    ("results/visual_localization_sdnet2018_decks.png", "SDNET2018 (Decks) Anomaly Localization Heatmaps"),
    ("results/visual_localization_deepcrack_cross-eval.png", "DeepCrack (Cross-Dataset) Anomaly Localization Heatmaps"),
]

for path, title in visuals:
    if os.path.exists(path):
        print(f"\n{'='*60}\n{title}\n{'='*60}")
        img = Image.open(path)
        plt.figure(figsize=(12, 10))
        plt.imshow(img)
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print(f"Visual overlay skipped (not found): {path}")